<a href="https://colab.research.google.com/github/metalurgista-4618/Balance_ejemplo/blob/main/AplicacionBalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from io import StringIO
from matplotlib.lines import lineStyles
from ipywidgets.widgets.widget_string import Label
from ipywidgets.widgets.widget_color import Color


## CREACION DE APLICACION CON INPUTS PARA EL BALANCE METALURGICO
Con esta aplicacion se podran tener las dieferntes tablas para cada numnero de malla, asi como tambien calcular en que punto es la recuperacion.
Luego cada dato de la recuperacion de la tabla se grabara a otra tabla para su posterior analisis de rendimineto en funcion del tamaño de malla vs recuperacion.
Asi como tambien el posterior analisis de exeperimentos

In [4]:
#1. Inicio de aplicacion
def datos_balance():

  global malla,w_bruto,w_concentrado,w_relave,Au_c,Au_r,\
  Au_cc,Ag_c,Ag_r,Ag_cc,porc_c,porc_r,m_au_c,m_au_r,m_ag_c,m_ag_r,\
  cont_fino_au_c,cont_fino_au_r,total_fino_au,rec_au_c,rec_au_r,\
  cont_fino_ag_c,  cont_fino_ag_r,total_fino_ag, rec_ag_c,rec_ag_r,Ratio


  malla=float(input('Escriba el numero de malla: '))
  #Partes
  w_bruto = float(input('Escriba el Peso Bruto ingresa al proceso: '))
  w_concentrado = float(input('Escriba el Peso Concentrado saliente del proceso: '))
  w_relave = float(input('Escriba el Peso Relave saliente del proceso: '))

  Au_c= float(input('Ingrese la Ley gr/TM de Au del concentrado: '))
  Au_r= float(input('Ingrese la Ley gr/TM de Au del relave: '))
  Au_cc= round((Au_c*w_concentrado+Au_r*w_relave)/w_bruto,2)
  Au_cc
  #Ag
  Ag_c= float(input('Ingrese la Ley gr/TM de Ag del concentrado: '))
  Ag_r= float(input('Ingrese la Ley gr/TM de Ag del relave: '))
  Ag_cc= round((Ag_c*w_concentrado+Ag_r*w_relave)/w_bruto,2)
  Ag_cc= round((Ag_c*w_concentrado+Ag_r*w_relave)/w_bruto,2)
  #%MAsa
  porc_c = round((w_concentrado/w_bruto)*100,2)
  porc_r = round((w_relave/w_bruto)*100,2)
  #Metalico
  #Au:
  m_au_c = round(w_concentrado*Au_c/w_bruto,2)
  m_au_r = round(w_relave*Au_r/w_bruto,2)
  #Ag:
  m_ag_c = round(w_concentrado*Ag_c/w_bruto,2)
  m_ag_r = round(w_relave*Ag_r/w_bruto,2)

  #%recuperacion
  #Au:
  cont_fino_au_c = w_concentrado*Au_c/1000000
  cont_fino_au_r = w_relave*Au_r/1000000
  total_fino_au = cont_fino_au_c + cont_fino_au_r
  rec_au_c = round((cont_fino_au_c/total_fino_au)*100,2)
  rec_au_r = 100-rec_au_c
  #Ag:
  cont_fino_ag_c = w_concentrado*Ag_c/1000000
  cont_fino_ag_r = w_relave*Ag_r/1000000
  total_fino_ag = cont_fino_ag_c + cont_fino_ag_r
  rec_ag_c = round((cont_fino_ag_c/total_fino_ag)*100,2)
  rec_ag_r = 100-rec_ag_c

  #Ratio de concentracion
  Ratio = round(w_bruto/w_concentrado,2)


In [5]:
#Armando la tabla de balance de materia
#Nota:  No se calcula la calidad gr/TM del mineral en Bruto ya que no es representativo
#hay falta de homogeneidad
def generar_balance():
  global dt_balance
  balance_metalurgico = {
      'PARTES':['MINERAL BRUTO','MINERAL CONCENTRADO','RELAVE','CABEZA CALCULADA'],
      'MASA (gr)': [w_bruto,w_concentrado,w_relave, '--'],
      'ORO(gr/TM)':['',Au_c,Au_r,Au_cc],
      'PLATA(gr/TM)':['',Ag_c,Ag_r,Ag_cc],
      'Metalico_Au':[100,m_au_c,m_au_r,''],
      'Metalico_Ag':['',m_ag_c,m_ag_r,''],
      'Rec_Au':['',rec_au_c,rec_au_r,''],
      'Rec_Ag':['',rec_ag_c,rec_ag_r,''],
      'RATIO':['',Ratio,'','']
  }
  dt_balance = pd.DataFrame(balance_metalurgico)
  print(f'El balance para malla {malla}, se tiene la sgte tabla: ')
  print('--------------------------------------------------------')
  return dt_balance

In [6]:
columnas = ['Malla','Recuperacion']
dt_registro_RT023 = pd.DataFrame(columns=columnas)

def grabar_tabla_au():
  global dt_registro_RT023
  fila = pd.DataFrame([{
      'Malla':malla,
      'Recuperacion':rec_au_c
  }])

  dt_registro_RT023 = pd.concat([dt_registro_RT023, fila], ignore_index=True)


In [7]:
#Exportar a EXcel o csv
def exportar_excel(nombredearchivo,dt_var):
    nombrefinal=f'{nombredearchivo}.xlsx'
    dt_var.to_excel(nombrefinal, index=False)

In [8]:
# Initialize a variable to store the DataFrame generated by generar_balance
current_dt_balance = None
current_table = {}
while True:
  print('Seleccione un numero')
  print('----------------------------')
  print('1. Ingresar datos')
  print('2. Generar Tabla')
  print('3. Exportar a excel')
  print('4. Grabar a Tabla RT023')
  print('5. Graficar puntos')
  print('6. Exportar a Excel')
  print('7. Salir')

  seleccion = int(input('Seleccione una opcion: '))
  if isinstance(seleccion, int):

    if seleccion == 1:
      datos_balance()
      # Reset the current balance if new data is entered, as the previous balance is no longer valid
      current_dt_balance = None
    if seleccion == 2:
      # Store the DataFrame returned by generar_balance
      current_dt_balance = generar_balance()
      # Display the stored DataFrame
      display(current_dt_balance)
    if seleccion == 3:
      # Check if a DataFrame has been generated
      if current_dt_balance is not None:
        Nombre = input('Especificar el nombre del archivo: ')
        exportar_excel(Nombre, current_dt_balance)
      else:
        print('Error: La tabla de balance no ha sido generada. Por favor, selecciona la opción 2 primero.')
    if seleccion == 4:
        print('Seleccionar si es Au o Ag como 1 o 2')
        digito = input('Escriba el numero')
        if digito == '1':
          current_table=grabar_tabla_au()
          display(dt_registro_RT023)
    if seleccion == 5:
      break
  else:
    print('Entrada inválida. Por favor, ingresa un número.')
    break


Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir
Seleccione una opcion: 1
Escriba el numero de malla: 31.82
Escriba el Peso Bruto ingresa al proceso: 2000
Escriba el Peso Concentrado saliente del proceso: 141.12
Escriba el Peso Relave saliente del proceso: 1852.1
Ingrese la Ley gr/TM de Au del concentrado: 35.2
Ingrese la Ley gr/TM de Au del relave: 6.1
Ingrese la Ley gr/TM de Ag del concentrado: 150.5
Ingrese la Ley gr/TM de Ag del relave: 100.4
Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir
Seleccione una opcion: 2
El balance para malla 31.82, se tiene la sgte tabla: 
--------------------------------------------------------


,PARTES,MASA (gr),ORO(gr/TM),PLATA(gr/TM),Metalico_Au,Metalico_Ag,Rec_Au,Rec_Ag,RATIO
0,MINERAL BRUTO,2000.0,,,100,,,,
1,MINERAL CONCENTRADO,141.12,35.2,150.5,2.48,10.62,30.54,10.25,14.17
2,RELAVE,1852.1,6.1,100.4,5.65,92.98,69.46,89.75,
3,CABEZA CALCULADA,--,8.13,103.59,,,,,


Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir
Seleccione una opcion: 4
Seleccionar si es Au o Ag como 1 o 2
Escriba el numero1


/tmp/ipykernel_22411/1087825224.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dt_registro_RT023 = pd.concat([dt_registro_RT023, fila], ignore_index=True)


,Malla,Recuperacion,RECUPERACION
0,31.82,NaN,30.54


Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir
Seleccione una opcion: 1
Escriba el numero de malla: 41.25
Escriba el Peso Bruto ingresa al proceso: 2000
Escriba el Peso Concentrado saliente del proceso: 133.9
Escriba el Peso Relave saliente del proceso: 1862.2
Ingrese la Ley gr/TM de Au del concentrado: 53.1
Ingrese la Ley gr/TM de Au del relave: 5.11
Ingrese la Ley gr/TM de Ag del concentrado: 190.22
Ingrese la Ley gr/TM de Ag del relave: 100.94
Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir
Seleccione una opcion: 4
Seleccionar si es Au o Ag como 1 o 2
Escriba el numero1


,Malla,Recuperacion,RECUPERACION
0,31.82,NaN,30.54
1,41.25,NaN,42.77


Seleccione un numero
----------------------------
1. Ingresar datos
2. Generar Tabla
3. Exportar a excel
4. Grabar a Tabla RT023
5. Salir


KeyboardInterrupt: Interrupted by user

In [ ]:
current_table